!! POT SER QUE FALLI SI L'ORDINADOR NO TÉ PROU MEMÒRIA RAM

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from utils.tobii import (
    compute_regression_metrics,
    detect_regressions,
    extract_fixations,
    get_hits,
    process_all_texts,
    read_all_data,
    read_data,
    z_score,
)

pd.set_option("display.max_columns", None)

In [ ]:
mused_chosen = pd.read_csv("../data/mused_chosen_data.csv")
mused_chosen["num_id"] = mused_chosen.id.str.split("_").str[1].astype(int)
texts = dict(zip(mused_chosen.num_id, mused_chosen.text_clean))
text_ids = mused_chosen.num_id.tolist()

In [ ]:
PROJECT = "../data/tobii/"
general = (
    pd.read_csv(PROJECT + "general.tsv", sep="\t")
    .drop(["Event", "Recording timestamp", "Computer timestamp"], axis=1)
    .drop_duplicates()
)
# general.head()

In [ ]:
aoi_hit, calibration_dfs, participants, dfs = read_all_data("../data/tobii/all_parquets")
part = list(dict.fromkeys([x[0] for x in participants]))  # unique names in order

In [ ]:
# dfs_part = [pd.concat(dfs[i * 4 : (i + 1) * 4]) for i in range(len(part))]

In [ ]:
# dfs té un df per fitxer de parquet
# text_dfs té un df per persona i per text
text_dfs = {}
aoi_cols_dict = {}
tfds = {}  # Total fixation durations dict user: text_id: tfds
for part_idx in range(len(part)):
    participant = part[part_idx]
    text_dfs_part, aoi_cols_part = process_all_texts(
        dfs[part_idx * 4 : (part_idx + 1) * 4], aoi_hit
    )
    text_dfs[participant] = text_dfs_part
    aoi_cols_dict[participant] = aoi_cols_part

    tfds[participant] = {}
    for text_id in text_ids:
        _df = text_dfs_part[text_id]
        # Total fixation duration
        all_aois = aoi_cols_part[text_id]
        tfd = _df[_df["event_type"] == "Fixation"].groupby(by="AOI")["duration"].sum()
        tfd = tfd.reindex(all_aois, fill_value=0).to_numpy()
        tfds[participant][text_id] = tfd / tfd.sum()

In [ ]:
tfds["karla"]

# MuSeD spans

In [ ]:
data = pd.read_csv("../data/chosen_data_full.csv")
data["num_id"] = data["id"].str.replace("video_", "").astype(int)
# data["label_clean"] =  data["label_clean"].apply(lambda x: eval(x))

In [ ]:
annotations = {}  # max-normalized (for filtering)
annotations_sum = {}  # sum-normalized (for comparison with eye-tracking)
per_label_annotations = {}  # {label: {text_id: sum-normalized array}}

for text_id in data.num_id.tolist():
    row = data[data.num_id == text_id].iloc[0]
    text = row["text_clean"]

    matches = list(re.finditer(r"\S+", text))
    tokens = [m.group() for m in matches]
    starts = [m.start() for m in matches]
    idx2token = dict(zip(starts, range(len(tokens))))

    labels = eval(row["label_clean"])
    _annotations = np.zeros(len(tokens))
    _per_label = {}  # label -> raw count array

    for annotation_id in range(len(labels)):
        start = labels[annotation_id]["start"]
        end = labels[annotation_id]["end"]
        while idx2token.get(start) is None:
            start += 1
        while idx2token.get(end) is None:
            end -= 1
        _annotations[idx2token[start] : idx2token[end] + 1] += 1

        for tag in labels[annotation_id]["labels"]:
            if tag not in _per_label:
                _per_label[tag] = np.zeros(len(tokens))
            _per_label[tag][idx2token[start] : idx2token[end] + 1] += 1

    # max normalization: useful to filter texts where most tokens are annotated
    _annotations_max = np.divide(_annotations.copy(), max(_annotations.max(), 1))
    annotations[text_id] = _annotations_max

    # sum normalization: probabilities for comparison with eye-tracking
    total = _annotations.sum()
    annotations_sum[text_id] = _annotations / max(total, 1)

    # per-label: sum-normalized
    for tag, raw in _per_label.items():
        if tag not in per_label_annotations:
            per_label_annotations[tag] = {}
        per_label_annotations[tag][text_id] = raw / max(raw.sum(), 1)

print(f"{len(per_label_annotations)} label types: {sorted(per_label_annotations.keys())}")

In [ ]:
# {k: (v, v.mean()) for k, v in sorted(annotations.items(), key=lambda item: item[1].mean())}

# Comparison: eye-tracking vs. span annotations

In [ ]:
# Filter out texts where all tokens are annotated (no discriminative signal)
annotated_texts = [
    tid for tid, v in annotations.items() if (v.mean() < 1.0 and v.mean() > 0)
]
print(f"{len(annotated_texts)} texts with partial annotations (out of {len(annotations)})")
print(f"Excluded (all annotated): {sorted(set(annotations) - set(annotated_texts))}")

In [ ]:
EPS = 1e-12  # avoid log(0)


def safe_cross_entropy(p, q):
    """Cross-entropy H(p, q) = -sum(p * log(q)), with epsilon for numerical stability."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    q = np.clip(q, EPS, None)
    return -np.sum(p * np.log(q))


def safe_kl_divergence(p, q):
    """KL(p || q) = sum(p * log(p / q)), with epsilon for numerical stability."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.clip(p, EPS, None)
    q = np.clip(q, EPS, None)
    return np.sum(p * np.log(p / q))


def jensen_shannon_divergence(p, q, base2=True):
    """JSD(p || q) = 0.5 * KL(p || m) + 0.5 * KL(q || m), where m = 0.5 * (p + q)."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    m = 0.5 * (p + q)
    jsd = 0.5 * safe_kl_divergence(p, m) + 0.5 * safe_kl_divergence(q, m)
    if base2:
        jsd /= np.log(2)
    return jsd


def compute_metrics(eye, ann):
    """Compute Spearman, cross-entropy, and KL-divergence for two aligned vectors."""
    n = min(len(eye), len(ann))
    eye, ann = eye[:n], ann[:n]
    if n == 0:
        return None
    rho, _ = stats.spearmanr(eye, ann)
    return {
        "spearman": rho,
        "cross_entropy": safe_cross_entropy(ann, eye),
        "kl_div": safe_kl_divergence(ann, eye),
        "js_div": jensen_shannon_divergence(ann, eye),
    }


# Compute per-participant metrics (overall)
metrics_records = []

for participant in part:
    for text_id in annotated_texts:
        if text_id not in tfds.get(participant, {}):
            continue
        m = compute_metrics(tfds[participant][text_id], annotations_sum[text_id])
        if m is not None:
            metrics_records.append({"participant": participant, "text_id": text_id, **m})

metrics_df = pd.DataFrame(metrics_records)
print(f"Computed {len(metrics_df)} (participant, text) pairs")

In [ ]:
# Per-participant summary
print("=== Spearman correlation (eye-tracking vs annotations, per participant) ===")
print(metrics_df.groupby("participant")["spearman"].agg(["mean", "std", "count"]))
print()
print("=== Cross-entropy H(annotation | eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["cross_entropy"].agg(["mean", "std", "count"]))
print()
print("=== KL-divergence KL(annotation || eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["kl_div"].agg(["mean", "std", "count"]))
print("=== JS-divergence JS(annotation || eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["js_div"].agg(["mean", "std", "count"]))

In [ ]:
# Averaged across participants per text
text_avg = metrics_df.groupby("text_id").agg(
    spearman_mean=("spearman", "mean"),
    spearman_std=("spearman", "std"),
    ce_mean=("cross_entropy", "mean"),
    ce_std=("cross_entropy", "std"),
    kl_mean=("kl_div", "mean"),
    kl_std=("kl_div", "std"),
    js_mean=("js_div", "mean"),
    js_std=("js_div", "std"),
    n_participants=("participant", "count"),
)
print("=== Per-text metrics (averaged across participants) ===")
print(text_avg.sort_values("spearman_mean", ascending=False).round(2).to_string())
print()
print(f"Overall mean Spearman rho: {text_avg['spearman_mean'].mean():.2f}")
print(f"Overall mean cross-entropy: {text_avg['ce_mean'].mean():.2f}")
print(f"Overall mean KL-divergence: {text_avg['kl_mean'].mean():.2f}")
print(f"Overall mean JS-divergence: {text_avg['js_mean'].mean():.2f}")

In [ ]:
# Visualization: per text, averaged across participants
gender_map = dict(zip(general["Participant name"], general["sexe"]))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))


axes[0].bar(range(len(text_avg)), text_avg["spearman_mean"])
axes[0].set_xlabel("Text ID")
axes[0].set_ylabel("Mean Spearman rho")
axes[0].set_title("Spearman correlation per text (avg across participants)")
axes[0].axhline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].bar(range(len(text_avg)), text_avg["ce_mean"])
axes[1].set_xlabel("Text ID")
axes[1].set_ylabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per text (avg across participants)")

axes[2].bar(range(len(text_avg)), text_avg["js_mean"])
axes[2].set_xlabel("Text ID")
axes[2].set_ylabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per text (avg across participants)")

plt.tight_layout()
plt.show()

In [ ]:
# Per-participant bar chart
part_summary = metrics_df.groupby("participant").agg(
    spearman_mean=("spearman", "mean"),
    ce_mean=("cross_entropy", "mean"),
    kl_mean=("kl_div", "mean"),
    js_mean=("js_div", "mean"),
)
part_summary["sex"] = part_summary.index.map(gender_map)
part_summary = part_summary.sort_values("spearman_mean", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ["#1f77b4" if s == "home" else "#e377c2" for s in part_summary["sex"]]

axes[0].barh(part_summary.index, part_summary["spearman_mean"], color=colors)
axes[0].set_xlabel("Mean Spearman rho")
axes[0].set_title("Spearman correlation per participant")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].barh(part_summary.index, part_summary["ce_mean"], color=colors)
axes[1].set_xlabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per participant")

axes[2].barh(part_summary.index, part_summary["js_mean"], color=colors)
axes[2].set_xlabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per participant")

plt.tight_layout()
plt.show()

# Per-label span analysis

In [ ]:
# Compute metrics per label type from span annotations
label_metrics_records = []

for tag, tag_anns in per_label_annotations.items():
    for participant in part:
        for text_id in annotated_texts:
            if text_id not in tfds.get(participant, {}):
                continue
            if text_id not in tag_anns:
                continue
            m = compute_metrics(tfds[participant][text_id], tag_anns[text_id])
            if m is not None:
                label_metrics_records.append(
                    {
                        "label": tag,
                        "participant": participant,
                        "text_id": text_id,
                        **m,
                    }
                )

label_metrics_df = pd.DataFrame(label_metrics_records)
print(
    f"Computed {len(label_metrics_df)} records across {label_metrics_df['label'].nunique()} label types"
)

In [ ]:
# Per-label summary (averaged across participants and texts)
print("=== Metrics per span label type ===")
print(
    label_metrics_df.groupby("label")
    .agg(
        spearman_mean=("spearman", "mean"),
        spearman_std=("spearman", "std"),
        ce_mean=("cross_entropy", "mean"),
        kl_mean=("kl_div", "mean"),
        js_mean=("js_div", "mean"),
        n=("text_id", "count"),
    )
    .sort_values("spearman_mean", ascending=False)
    .round(2)
    .to_string()
)

In [ ]:
# Visualize per-label metrics
label_summary = (
    label_metrics_df.groupby("label")
    .agg(spearman=("spearman", "mean"), ce=("cross_entropy", "mean"), js=("js_div", "mean"))
    .sort_values("spearman", ascending=True)
)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

axes[0].barh(label_summary.index, label_summary["spearman"])
axes[0].set_xlabel("Mean Spearman rho")
axes[0].set_title("Spearman per label type")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].barh(label_summary.index, label_summary["ce"])
axes[1].set_xlabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per label type")

axes[2].barh(label_summary.index, label_summary["js"])
axes[2].set_xlabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per label type")

plt.tight_layout()
plt.show()

# Per-binary-column analysis (text-level categories)

In [ ]:
# Compare metrics split by each binary text-level column
label_cols = [
    "sexist",
    "irony",
    "humor",
    "implicit sexism",
    "stereotypes",
    "inequality",
    "discrimination",
    "objectification",
    "critique",
    "reported_sexism",
]

split_metrics_records = []
for col in label_cols:
    for _, row in data.iterrows():
        text_id = row["num_id"]
        group = row[col]
        for participant in part:
            if text_id not in tfds.get(participant, {}):
                continue
            if text_id not in annotations_sum:
                continue
            m = compute_metrics(tfds[participant][text_id], annotations_sum[text_id])
            if m is not None:
                split_metrics_records.append(
                    {
                        "column": col,
                        "group": int(group),
                        "participant": participant,
                        "text_id": text_id,
                        **m,
                    }
                )

split_df = pd.DataFrame(split_metrics_records)
print(f"Computed {len(split_df)} records across {split_df['column'].nunique()} columns")

In [ ]:
# Summary: mean metrics per column, per group
summary = (
    split_df.groupby(["column", "group"])
    .agg(
        spearman=("spearman", "mean"),
        cross_entropy=("cross_entropy", "mean"),
        js_div=("js_div", "mean"),
        n_texts=("text_id", "nunique"),
        n_pairs=("text_id", "count"),
    )
    .unstack(level="group")
    .round(2)
)

# Flatten multi-level columns
summary.columns = ["_".join(str(c) for c in col).strip("_") for col in summary.columns]
print("=== Metrics split by text-level binary categories ===")
print(summary.to_string())

In [ ]:
# Visualize the delta (positive - negative) for each category
deltas = []
for col in label_cols:
    sub = split_df[split_df["column"] == col]
    pos = sub[sub["group"] == 1]
    neg = sub[sub["group"] == 0]
    if len(pos) > 0 and len(neg) > 0:
        for metric in ["spearman", "cross_entropy", "js_div"]:
            deltas.append(
                {
                    "column": col,
                    "metric": metric,
                    "mean_positive": pos[metric].mean(),
                    "mean_negative": neg[metric].mean(),
                    "delta": pos[metric].mean() - neg[metric].mean(),
                }
            )

delta_df = pd.DataFrame(deltas)
print("Delta (positive group - negative group) per category:")
print(delta_df.pivot(index="column", columns="metric", values="delta").round(2).to_string())

In [ ]:
# Bar chart of deltas per category
pivot = delta_df.pivot(index="column", columns="metric", values="delta")

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, metric in zip(axes, ["spearman", "cross_entropy", "js_div"]):
    vals = pivot[metric].sort_values()
    colors = ["#2ca02c" if v > 0 else "#d62728" for v in vals]
    ax.barh(vals.index, vals, color=colors)
    ax.set_xlabel(f"Delta {metric}")
    ax.set_title(f"Delta {metric} (has label - no label)")
    ax.axvline(0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# Regression analysis

In [ ]:
# Compute regressions for all participant × text pairs
all_regressions = []
all_reg_metrics = []

for participant in list(text_dfs.keys()):
    for text_id in text_ids:
        if text_id not in text_dfs.get(participant, {}):
            continue
        df = text_dfs[participant][text_id]
        fixations = extract_fixations(df)
        regressions = detect_regressions(fixations)
        metrics = compute_regression_metrics(fixations, regressions)
        if len(regressions) > 0:
            regressions = regressions.copy()
            regressions["participant"] = participant
            regressions["text_id"] = text_id
            all_regressions.append(regressions)
        metrics["participant"] = participant
        metrics["text_id"] = text_id
        all_reg_metrics.append(metrics)

all_regressions_df = pd.concat(all_regressions, ignore_index=True)
all_reg_metrics_df = pd.DataFrame(all_reg_metrics)

# Extract token indices from AOI column names
all_regressions_df["target_idx"] = (
    all_regressions_df["to_AOI"].str.split(".", n=1).str[0].astype(int)
)
all_regressions_df["source_idx"] = (
    all_regressions_df["from_AOI"].str.split(".", n=1).str[0].astype(int)
)

print(f"Total regressions: {len(all_regressions_df)}")
print(
    f"Regressions per text (mean ± std): "
    f"{all_reg_metrics_df.groupby('text_id')['total_regressions'].sum().mean():.1f} "
    f"± {all_reg_metrics_df.groupby('text_id')['total_regressions'].sum().std():.1f}"
)
print("\nRegressions by type:")
print(all_regressions_df["regression_type"].value_counts())

In [ ]:
n_bins = 10
participant_baselines = {}

for participant in list(text_dfs.keys()):
    p_mets = all_reg_metrics_df[all_reg_metrics_df["participant"] == participant]
    p_regs = all_regressions_df[all_regressions_df["participant"] == participant]

    total_saccades = p_mets["total_saccades"].sum()
    total_regressions = p_mets["total_regressions"].sum()
    reg_rate = total_regressions / total_saccades if total_saccades > 0 else 0

    # Normalized target positions
    target_positions = []
    source_positions = []
    for _, row in p_regs.iterrows():
        n_tok = len(texts[row["text_id"]].split())
        target_positions.append(row["target_idx"] / max(n_tok - 1, 1))
        source_positions.append(row["source_idx"] / max(n_tok - 1, 1))

    target_hist, _ = np.histogram(target_positions, bins=n_bins, range=(0, 1))
    target_probs = target_hist / max(target_hist.sum(), 1)

    source_hist, _ = np.histogram(source_positions, bins=n_bins, range=(0, 1))
    source_probs = source_hist / max(source_hist.sum(), 1)

    participant_baselines[participant] = {
        "regression_rate": reg_rate,
        "total_regressions": total_regressions,
        "total_saccades": total_saccades,
        "mean_dAOI": p_regs["dAOI"].mean() if len(p_regs) > 0 else 0,
        "std_dAOI": p_regs["dAOI"].std() if len(p_regs) > 0 else 0,
        "target_probs": target_probs,
        "source_probs": source_probs,
    }

print("Per-participant baselines:")
for p, bl in participant_baselines.items():
    print(
        f"  {p}: rate={bl['regression_rate']:.3f}, "
        f"mean_dAOI={bl['mean_dAOI']:.1f}, "
        f"n_regressions={bl['total_regressions']}"
    )

In [ ]:
def bh_fdr(p_values):
    """Benjamini-Hochberg FDR correction."""
    n = len(p_values)
    ranked = np.argsort(p_values)  # smallest first
    adjusted = np.zeros(n)
    # Compute raw adjusted p-values
    for i, idx in enumerate(ranked):
        adjusted[idx] = min(p_values[idx] * n / (i + 1), 1.0)
    # Enforce monotonicity: cumulative min from largest to smallest
    cummin = 1.0
    for i in range(n - 1, -1, -1):
        idx = ranked[i]
        cummin = min(cummin, adjusted[idx])
        adjusted[idx] = cummin
    return adjusted


zscore_target_records = []

for text_id in text_ids:
    n_tokens = len(texts[text_id].split())
    participants_reading = [p for p in part if text_id in text_dfs.get(p, {})]
    for token_idx in range(n_tokens):
        obs_total = 0
        exp_total = 0
        for participant in participants_reading:
            obs = len(
                all_regressions_df[
                    (all_regressions_df["participant"] == participant)
                    & (all_regressions_df["text_id"] == text_id)
                    & (all_regressions_df["target_idx"] == token_idx)
                ]
            )

            bl = participant_baselines[participant]
            rel_pos = token_idx / max(n_tokens - 1, 1)
            bin_idx = min(int(rel_pos * n_bins), n_bins - 1)

            n_sacc = all_reg_metrics_df[
                (all_reg_metrics_df["participant"] == participant)
                & (all_reg_metrics_df["text_id"] == text_id)
            ]["total_saccades"].values
            n_sacc = n_sacc[0] if len(n_sacc) > 0 else 0
            exp = bl["regression_rate"] * n_sacc * bl["target_probs"][bin_idx]

            obs_total += obs
            exp_total += exp

        z = (obs_total - exp_total) / np.sqrt(max(exp_total, 1e-6))
        zscore_target_records.append(
            {
                "text_id": text_id,
                "token_idx": token_idx,
                "token": texts[text_id].split()[token_idx],
                "observed": obs_total,
                "expected": round(exp_total, 2),
                "z_score": round(z, 3),
                "n_participants": len(participants_reading),
            }
        )

zscore_target_df = pd.DataFrame(zscore_target_records)

# BH correction
zscore_target_df["p_value"] = 2 * (1 - stats.norm.cdf(np.abs(zscore_target_df["z_score"])))
zscore_target_df["q_value"] = bh_fdr(zscore_target_df["p_value"].values)

salient_target_tokens = zscore_target_df[zscore_target_df["q_value"] < 0.05].sort_values(
    "z_score", ascending=False
)

print(f"Salient TARGET tokens (BH q < 0.05): {len(salient_target_tokens)}")
print("\nTop 20:")
salient_target_tokens.head(20)

In [ ]:
zscore_source_records = []

for text_id in text_ids:
    n_tokens = len(texts[text_id].split())
    participants_reading = [p for p in part if text_id in text_dfs.get(p, {})]
    for token_idx in range(n_tokens):
        obs_total = 0
        exp_total = 0
        for participant in participants_reading:
            obs = len(
                all_regressions_df[
                    (all_regressions_df["participant"] == participant)
                    & (all_regressions_df["text_id"] == text_id)
                    & (all_regressions_df["source_idx"] == token_idx)
                ]
            )

            bl = participant_baselines[participant]
            rel_pos = token_idx / max(n_tokens - 1, 1)
            bin_idx = min(int(rel_pos * n_bins), n_bins - 1)

            n_sacc = all_reg_metrics_df[
                (all_reg_metrics_df["participant"] == participant)
                & (all_reg_metrics_df["text_id"] == text_id)
            ]["total_saccades"].values
            n_sacc = n_sacc[0] if len(n_sacc) > 0 else 0
            exp = bl["regression_rate"] * n_sacc * bl["source_probs"][bin_idx]

            obs_total += obs
            exp_total += exp

        z = (obs_total - exp_total) / np.sqrt(max(exp_total, 1e-6))
        zscore_source_records.append(
            {
                "text_id": text_id,
                "token_idx": token_idx,
                "token": texts[text_id].split()[token_idx],
                "observed": obs_total,
                "expected": round(exp_total, 2),
                "z_score": round(z, 3),
                "n_participants": len(participants_reading),
            }
        )

zscore_source_df = pd.DataFrame(zscore_source_records)

# BH correction
zscore_source_df["p_value"] = 2 * (1 - stats.norm.cdf(np.abs(zscore_source_df["z_score"])))
zscore_source_df["q_value"] = bh_fdr(zscore_source_df["p_value"].values)

salient_source_tokens = zscore_source_df[zscore_source_df["q_value"] < 0.05].sort_values(
    "z_score", ascending=False
)

print(f"Salient SOURCE tokens (BH q < 0.05): {len(salient_source_tokens)}")
print("\nTop 20:")
salient_source_tokens.head(20)

In [ ]:
# Salient targets per text
print("=== Salient TARGET tokens per text ===")
salient_tgt_per_text = (
    salient_target_tokens.groupby("text_id")
    .agg(
        n_salient=("token_idx", "count"),
        tokens=("token", list),
        max_z=("z_score", "max"),
    )
    .sort_values("max_z", ascending=False)
)
print(salient_tgt_per_text.to_string())

print("\n=== Salient SOURCE tokens per text ===")
salient_src_per_text = (
    salient_source_tokens.groupby("text_id")
    .agg(
        n_salient=("token_idx", "count"),
        tokens=("token", list),
        max_z=("z_score", "max"),
    )
    .sort_values("max_z", ascending=False)
)
print(salient_src_per_text.to_string())

In [ ]:
# Heatmap: z-score for targets and sources across all texts
# Show texts on y-axis, token position (normalized) on x-axis
texts_to_show = sorted(set(zscore_target_df["text_id"]))

fig, axes = plt.subplots(1, 2, figsize=(20, max(8, len(texts_to_show) * 0.4)))

for ax, df, title in [
    (axes[0], zscore_target_df, "Target z-score (regressions TO)"),
    (axes[1], zscore_source_df, "Source z-score (regressions FROM)"),
]:
    # Build matrix: texts × max token length
    max_tok = max(len(texts[t].split()) for t in texts_to_show)
    mat = np.full((len(texts_to_show), max_tok), np.nan)

    for row in df.itertuples():
        t_idx = texts_to_show.index(row.text_id)
        mat[t_idx, row.token_idx] = row.z_score

    im = ax.imshow(
        mat, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest"
    )
    ax.set_yticks(range(len(texts_to_show)))
    ax.set_yticklabels(texts_to_show, fontsize=7)
    ax.set_xlabel("Token position (normalized)")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="z-score")

plt.tight_layout()
plt.show()

In [ ]:
import time

# Pre-compute AOI index arrays for fast shuffling (avoids string ops in permutation loop)
pre_arrays = {}
for participant in list(text_dfs.keys()):
    for text_id in text_ids:
        if text_id not in text_dfs.get(participant, {}):
            continue
        fixations = extract_fixations(text_dfs[participant][text_id])
        fix_with_aoi = fixations.dropna(subset=["AOI"])
        aoi_idx = fix_with_aoi["AOI"].str.split(".", n=1).str[0].astype(int).values
        pre_arrays[(participant, text_id)] = aoi_idx

text_ids_with_data = [
    t for t in text_ids if any((p, t) in pre_arrays for p in list(text_dfs.keys()))
]
print(f"Pre-computed AOI arrays for {len(pre_arrays)} participant × text pairs")

# Benchmark one permutation iteration
t0 = time.time()
for perm in range(10):
    for text_id in text_ids_with_data:
        for participant in list(text_dfs.keys()):
            key = (participant, text_id)
            if key not in pre_arrays:
                continue
            aoi_idx = pre_arrays[key]
            perm_aoi_idx = aoi_idx[np.random.permutation(len(aoi_idx))]
            _ = np.diff(perm_aoi_idx)
elapsed_per_perm = (time.time() - t0) / 10
n_perms = 10000
est_minutes = elapsed_per_perm * n_perms / 60
print(
    f"One permutation: {elapsed_per_perm:.2f}s → {n_perms} permutations: ~{est_minutes:.1f} min"
)

In [ ]:
# Observed target and source counts per text × token
obs_target_counts = {}
obs_source_counts = {}
for text_id in text_ids_with_data:
    n_tokens = len(texts[text_id].split())
    target_counts = np.zeros(n_tokens)
    source_counts = np.zeros(n_tokens)
    regs = all_regressions_df[all_regressions_df["text_id"] == text_id]
    for idx in regs["target_idx"]:
        if 0 <= idx < n_tokens:
            target_counts[idx] += 1
    for idx in regs["source_idx"]:
        if 0 <= idx < n_tokens:
            source_counts[idx] += 1
    obs_target_counts[text_id] = target_counts
    obs_source_counts[text_id] = source_counts

# Permutation test for TARGET regressions (optimized)
n_perms = 10000
null_target_counts = {t: [] for t in text_ids_with_data}
null_source_counts = {t: [] for t in text_ids_with_data}

t_start = time.time()
for perm in range(n_perms):
    for text_id in text_ids_with_data:
        n_tokens = len(texts[text_id].split())
        target_counts = np.zeros(n_tokens)
        source_counts = np.zeros(n_tokens)
        for participant in list(text_dfs.keys()):
            key = (participant, text_id)
            if key not in pre_arrays:
                continue
            aoi_idx = pre_arrays[key]
            perm_aoi_idx = aoi_idx[np.random.permutation(len(aoi_idx))]
            dAOI = np.diff(perm_aoi_idx)
            reg_mask = dAOI < 0
            targets = perm_aoi_idx[1:][reg_mask]
            sources = perm_aoi_idx[:-1][reg_mask]
            for t_idx in targets:
                if 0 <= t_idx < n_tokens:
                    target_counts[t_idx] += 1
            for s_idx in sources:
                if 0 <= s_idx < n_tokens:
                    source_counts[s_idx] += 1
        null_target_counts[text_id].append(target_counts)
        null_source_counts[text_id].append(source_counts)
    if (perm + 1) % 1000 == 0:
        elapsed = time.time() - t_start
        rate = (perm + 1) / elapsed
        remaining = (n_perms - perm - 1) / rate
        print(
            f"  Perm {perm + 1}/{n_perms}: {elapsed:.1f}s elapsed, ~{remaining:.0f}s remaining"
        )

elapsed_total = time.time() - t_start
print(f"\nPermutation test done in {elapsed_total:.1f}s")

In [ ]:
# Compute permutation p-values per text × token
perm_target_pvalues = {}
perm_source_pvalues = {}
for text_id in text_ids_with_data:
    null_arr = np.array(null_target_counts[text_id])
    obs = obs_target_counts[text_id]
    perm_target_pvalues[text_id] = (null_arr >= obs[np.newaxis, :]).mean(axis=0)

    null_arr = np.array(null_source_counts[text_id])
    obs = obs_source_counts[text_id]
    perm_source_pvalues[text_id] = (null_arr >= obs[np.newaxis, :]).mean(axis=0)

# BH correction across all tokens in all texts
all_pvals_target = np.concatenate([perm_target_pvalues[t] for t in text_ids_with_data])
all_qvals_target = bh_fdr(all_pvals_target)
perm_target_qvalues = {}
offset = 0
for text_id in text_ids_with_data:
    n = len(perm_target_pvalues[text_id])
    perm_target_qvalues[text_id] = all_qvals_target[offset : offset + n]
    offset += n

all_pvals_source = np.concatenate([perm_source_pvalues[t] for t in text_ids_with_data])
all_qvals_source = bh_fdr(all_pvals_source)
perm_source_qvalues = {}
offset = 0
for text_id in text_ids_with_data:
    n = len(perm_source_pvalues[text_id])
    perm_source_qvalues[text_id] = all_qvals_source[offset : offset + n]
    offset += n

perm_sig_targets = sum(
    1 for t in text_ids_with_data for q in perm_target_qvalues[t] if q < 0.05
)
perm_sig_sources = sum(
    1 for t in text_ids_with_data for q in perm_source_qvalues[t] if q < 0.05
)
print(f"Permutation-significant TARGET tokens (q < 0.05): {perm_sig_targets}")
print(f"Permutation-significant SOURCE tokens (q < 0.05): {perm_sig_sources}")

In [ ]:
# Compare z-score vs permutation: TARGET tokens
z_target_sig = set(
    zip(
        salient_target_tokens["text_id"],
        salient_target_tokens["token_idx"],
    )
)
perm_target_sig = set()
for text_id in text_ids_with_data:
    for idx, q in enumerate(perm_target_qvalues[text_id]):
        if q < 0.05:
            perm_target_sig.add((text_id, idx))

both_target = z_target_sig & perm_target_sig
z_only_target = z_target_sig - perm_target_sig
perm_only_target = perm_target_sig - z_target_sig

print("=== TARGET tokens (regressions TO) ===")
print(f"  z-score only: {len(z_only_target)}")
print(f"  permutation only: {len(perm_only_target)}")
print(f"  both: {len(both_target)}")

# Compare z-score vs permutation: SOURCE tokens
z_source_sig = set(
    zip(
        salient_source_tokens["text_id"],
        salient_source_tokens["token_idx"],
    )
)
perm_source_sig = set()
for text_id in text_ids_with_data:
    for idx, q in enumerate(perm_source_qvalues[text_id]):
        if q < 0.05:
            perm_source_sig.add((text_id, idx))

both_source = z_source_sig & perm_source_sig
z_only_source = z_source_sig - perm_source_sig
perm_only_source = perm_source_sig - z_source_sig

print("\n=== SOURCE tokens (regressions FROM) ===")
print(f"  z-score only: {len(z_only_source)}")
print(f"  permutation only: {len(perm_only_source)}")
print(f"  both: {len(both_source)}")

# Combined (either target or source significant by either method)
all_sig_tokens = z_target_sig | perm_target_sig | z_source_sig | perm_source_sig
print(
    f"\nTotal unique (text_id, token_idx) significant by any method: {len(all_sig_tokens)}"
)

In [ ]:
from scipy.stats import fisher_exact

# For each text, check overlap of significant tokens with annotated spans
# Use the union of target-significant and source-significant tokens
all_sig_by_text = {}
for text_id, token_idx in all_sig_tokens:
    all_sig_by_text.setdefault(text_id, set()).add(token_idx)

overlap_records = []
fisher_results = []

for text_id in sorted(all_sig_by_text):
    if text_id not in annotations:
        continue
    ann = annotations[text_id]
    n_tokens = len(ann)
    sig_toks = all_sig_by_text[text_id]

    # Build 2x2 table
    a = sum(1 for i in sig_toks if i < n_tokens and ann[i] > 0)  # sig + in span
    b = sum(1 for i in sig_toks if i < n_tokens and ann[i] == 0)  # sig + not in span
    not_sig = set(range(n_tokens)) - sig_toks
    c = sum(1 for i in not_sig if i < n_tokens and ann[i] > 0)  # not sig + in span
    d = sum(1 for i in not_sig if i < n_tokens and ann[i] == 0)  # not sig + not in span

    for i in sorted(sig_toks):
        if i < n_tokens:
            overlap_records.append(
                {
                    "text_id": text_id,
                    "token_idx": i,
                    "token": texts[text_id].split()[i],
                    "in_span": ann[i] > 0,
                    "annotation_value": round(float(ann[i]), 3),
                }
            )

    if (a + b) > 0 and (c + d) > 0:
        odds, p = fisher_exact([[a, b], [c, d]], alternative="greater")
        fisher_results.append(
            {
                "text_id": text_id,
                "a_sig_in_span": a,
                "b_sig_not_in_span": b,
                "c_not_sig_in_span": c,
                "d_not_sig_not_in_span": d,
                "odds_ratio": odds,
                "p_value": p,
            }
        )

overlap_df = pd.DataFrame(overlap_records)
fisher_df = pd.DataFrame(fisher_results)

total_sig_in_span = overlap_df["in_span"].sum()
total_sig = len(overlap_df)
print(
    f"Significant tokens inside annotated spans: {total_sig_in_span}/{total_sig} "
    f"({100 * total_sig_in_span / max(total_sig, 1):.1f}%)"
)

print(f"\nFisher's exact test per text:")
print(fisher_df.to_string(index=False) if len(fisher_df) > 0 else "No results")

if len(fisher_df) > 0:
    sig_texts = fisher_df[fisher_df["p_value"] < 0.05]
    print(f"\nTexts with significant overlap (p < 0.05): {len(sig_texts)}")
    if len(sig_texts) > 0:
        print(
            sig_texts[
                ["text_id", "a_sig_in_span", "b_sig_not_in_span", "odds_ratio", "p_value"]
            ].to_string(index=False)
        )

In [ ]:
# Binary label comparison: regression rates for texts with vs without each label
binary_cols = [
    "sexist",
    "irony",
    "humor",
    "implicit sexism",
    "stereotypes",
    "inequality",
    "discrimination",
    "objectification",
    "critique",
    "reported_sexism",
]

binary_regression_results = []

for col in binary_cols:
    if col not in data.columns:
        continue
    group_1_ids = set(data[data[col] == 1]["num_id"])
    group_0_ids = set(data[data[col] == 0]["num_id"])

    for direction, idx_col in [("TO_target", "target_idx"), ("FROM_source", "source_idx")]:
        rates_1 = []
        rates_0 = []

        for participant in list(text_dfs.keys()):
            for text_id in text_ids:
                if text_id not in text_dfs.get(participant, {}):
                    continue
                m = all_reg_metrics_df[
                    (all_reg_metrics_df["participant"] == participant)
                    & (all_reg_metrics_df["text_id"] == text_id)
                ]
                if len(m) == 0:
                    continue
                rate = m["regression_rate"].values[0]
                if text_id in group_1_ids:
                    rates_1.append(rate)
                elif text_id in group_0_ids:
                    rates_0.append(rate)

        if len(rates_1) > 0 and len(rates_0) > 0:
            stat, p = stats.mannwhitneyu(rates_1, rates_0, alternative="two-sided")
            binary_regression_results.append(
                {
                    "column": col,
                    "direction": direction,
                    "mean_with": np.mean(rates_1),
                    "mean_without": np.mean(rates_0),
                    "delta": np.mean(rates_1) - np.mean(rates_0),
                    "U": stat,
                    "p": p,
                }
            )

binary_reg_df = pd.DataFrame(binary_regression_results)
print("=== Binary labels: regression rate comparison (Mann-Whitney U) ===")
print(binary_reg_df.to_string(index=False))

In [ ]:
# Span label types: compare z-scores inside vs outside each label type
span_label_results = []

for tag in sorted(per_label_annotations.keys()):
    in_z_scores = []
    out_z_scores = []

    for text_id, tag_ann in per_label_annotations[tag].items():
        text_z_target = zscore_target_df[zscore_target_df["text_id"] == text_id]
        text_z_source = zscore_source_df[zscore_source_df["text_id"] == text_id]

        in_tok_mask = tag_ann > 0
        out_tok_mask = tag_ann == 0

        for zdf, direction in [(text_z_target, "TO"), (text_z_source, "FROM")]:
            in_z = zdf[zdf["token_idx"].isin(np.where(in_tok_mask)[0])]["z_score"]
            out_z = zdf[zdf["token_idx"].isin(np.where(out_tok_mask)[0])]["z_score"]
            if len(in_z) > 0 and len(out_z) > 0:
                stat, p = stats.mannwhitneyu(in_z, out_z, alternative="greater")
                span_label_results.append(
                    {
                        "label": tag,
                        "direction": direction,
                        "text_id": text_id,
                        "mean_in": in_z.mean(),
                        "mean_out": out_z.mean(),
                        "p": p,
                    }
                )

span_label_df = pd.DataFrame(span_label_results)

# Aggregate per label type
span_agg = (
    span_label_df.groupby(["label", "direction"])
    .agg(
        mean_in=("mean_in", "mean"),
        mean_out=("mean_out", "mean"),
        n_texts=("text_id", "count"),
        n_sig=("p", lambda x: (x < 0.05).sum()),
    )
    .reset_index()
)

print("=== Span label types: mean z-score inside vs outside spans ===")
print(span_agg.to_string(index=False))

In [ ]:
# Visualize regression rate deltas per binary label
if len(binary_reg_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, direction in zip(axes, ["TO_target", "FROM_source"]):
        sub = binary_reg_df[binary_reg_df["direction"] == direction]
        if len(sub) == 0:
            continue
        vals = sub.set_index("column")["delta"].sort_values()
        colors = ["#2ca02c" if v > 0 else "#d62728" for v in vals]
        ax.barh(vals.index, vals, color=colors)
        ax.set_xlabel("Delta regression rate (with label - without)")
        ax.set_title(f"Regressions {direction}")
        ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

# continues

In [ ]:
# tsv_file = "sexism2 Recording3.tsv"
tsv_file = "sexism4 Data export"
aoi_hit, calibration_df, participants, df = read_data(tsv_file)

In [ ]:
text_dfs, aoi_cols_dict = process_all_texts(df, aoi_hit)

In [ ]:
text_df = text_dfs["2"]
aoi_cols = aoi_cols_dict["2"]

In [ ]:
fixations = extract_fixations(text_df)
fixations

In [ ]:
regressions = detect_regressions(fixations)
metrics = compute_regression_metrics(fixations, regressions)
regressions

# Other

In [ ]:
# això té sentit si deixem totes les columnes d'AOI
if aoi_hit[0] in text_df.columns:
    aoi_data = text_df[aoi_cols].fillna(0)

    # Check for simultaneous hits
    hits_per_row = aoi_data.sum(axis=1)
    multi_hits = hits_per_row[hits_per_row > 1]

    text_df.loc[multi_hits.index].drop(set(aoi_cols) - set(aoi_cols[152:154]), axis=1)

In [ ]:
# df[~df.Event.isna()]

In [ ]:
# raw_df[~raw_df.Event.isna()]

In [ ]:
# df[["gaze_x", "gaze_y"]]

In [ ]:
# df[df[aoi_hit[0]] != 0]

# Pupil diameter

In [ ]:
# hits to int
df = z_score(df, "Pupil diameter filtered", "participant")

In [ ]:
text_hits = get_hits(df, aoi_hit, "participant", normalize=True)

In [ ]:
# todo: is there a "text" column?, else -> create it
# -> unique [text, participant] to check if they did all the texts.

In [ ]:
texts = {t.split(" - ")[1].rsplit("]", 1)[0] for t in aoi_hit}
# texts = [t.split("[")[1].split()[0] for t in aoi_hit] # old?
text = list(texts)[2]
print(text)
plt.scatter(text_hits[text]["hits"], text_hits[text]["z_pupil"])
plt.show()